# Benchmark del calor 2D: PDNO frente a FNO

El operador solucion $u_0\mapsto u(t)$ del calor en $\mathbb T^2$, $a(k)=|k|^2$ (`HeatSymbol`, $n=2$), con los datos de `07_pdno_calor_2d.ipynb`.

Se comparan, sobre los mismos datos:

1. el **PDNO** con simbolos parabolicos en la configuracion lineal ($d=1$, sin $W$ ni sesgos, $\sigma=$ identidad, $P=Q=$ identidad, $L=1$), el elegido en el notebook PDNO, sin reentrenar;
2. un **FNO** de `neuraloperator` (version 2.0.0), con su propio barrido.

Las secciones son las de `10_benchmark_calor_1d.ipynb` sin la comparacion con el paper ni su Figura 3 (el paper solo tiene el calor 1D). Los dos modelos usan la misma perdida, el mismo optimizador y el mismo early stopping; la unica diferencia es la arquitectura.

# Setup
El codigo de cada seccion esta en `parabolic_symbol.learning.benchmark_experiment.BenchmarkExperiment`.

- **Datos:** los de `07_pdno_calor_2d.ipynb`, malla $128\times128$; $80$ $u_0$ para entrenar y $20$ para validar, y las $20$ de prueba en `TEST_TIMES`.
- **Precision:** `float32` en todo. El FNO de `neuraloperator` solo funciona en float32.
- **PDNO:** el `modelo.pt` de `07_pdno_calor_2d.ipynb`; hay que correr ese notebook antes.
- **FNO:** `train_baseline` con la `TRAIN_CONFIG` del PDNO y weight decay $0$; barrido `FNO_SWEEP` ($4\times2\times2=16$ corridas, `n_modes` por eje) y seleccion por menor $L^2$ relativa de validacion.

La celda de parametros reune todo lo configurable.

In [ ]:
from pathlib import Path

import numpy as np
import torch

from parabolic_symbol.symbols.heat import HeatSymbol
from parabolic_symbol.operators.fno_baseline import FNOConfig
from parabolic_symbol.learning.benchmark_experiment import BenchmarkExperiment
from parabolic_symbol.learning.operator_training import OperatorTrainingConfig
from parabolic_symbol.utils import default_device

In [ ]:
# ---------------------------------------------------------------- parametros
# Precision y device: float32 en todo (datos, PDNO, FNO, evaluacion y modelos guardados).
DEVICE = default_device()
DTYPE = torch.float32

# Datos: los de 07_pdno_calor_2d.ipynb (convencion del proyecto).
REFERENCE = HeatSymbol(n=2)    # a(k) = |k|^2
M = 2.0                        # orden del simbolo
RESOLUTION = 128               # malla 128 x 128
K_CAP = 32                     # k_max = k_cap
SEED = 0                       # 100 u0 de entrenamiento (TRAIN_TIMES) y 20 de prueba (TEST_TIMES)
VAL_FRACTION = 0.2             # fraccion de las u0 de entrenamiento que va a validacion

# Extrapolacion temporal: t en (0.95, 3], con las u0 de prueba; solo se reporta.
EXTRAPOLATION_TIMES = np.round(np.linspace(1.0, 3.0, 21), 10)

# PDNO: no se reentrena; se carga el modelo elegido en 07_pdno_calor_2d.ipynb.
PDNO_RESULTS = Path("../results/07_pdno_calor_2d")

# Entrenamiento: la TRAIN_CONFIG de 07_pdno_calor_2d.ipynb (se comprueba contra la del PDNO guardado). El FNO la usa con
# weight decay 0; sus campos del simbolo (omega, lambdas, C_alpha, ...) no se usan.
TRAIN_CONFIG = OperatorTrainingConfig(k_max=K_CAP, class_C_source="data", patience=60)
MODEL_SEED = 0

# FNO: barrido de n_modes (por eje) x hidden_channels x n_layers; se elige la menor L2 relativa de validacion.
FNO_BASE = FNOConfig(n_modes=16, hidden_channels=32, n_layers=2, positional_embedding="grid", seed=MODEL_SEED)
FNO_SWEEP = dict(n_modes=(8, 16, 32, 64), hidden_channels=(32, 64), n_layers=(2, 4))

# Invariancia de resolucion.
EVAL_RESOLUTIONS = (64, 128, 256)
RESOLUTION_EVAL_BATCH = 16      # lote de evaluacion en otras mallas: 256 x 256 con 64 no cabe en 6 GB

RESULTS = Path("../results/11_benchmark_calor_2d")   # tablas (CSV) y figuras (PNG)
RESULTS.mkdir(parents=True, exist_ok=True)
print(f"device = {DEVICE}  |  dtype = {DTYPE}")

In [ ]:
calor = BenchmarkExperiment(
    reference=REFERENCE, m=M, resolution=RESOLUTION, k_cap=K_CAP, results=RESULTS, pdno_results=PDNO_RESULTS,
    pdno_notebook="07_pdno_calor_2d.ipynb", train_config=TRAIN_CONFIG, dtype=DTYPE,
    fno_base=FNO_BASE, fno_sweep=FNO_SWEEP, extrapolation_times=EXTRAPOLATION_TIMES,
    eval_resolutions=EVAL_RESOLUTIONS, device=DEVICE, seed=SEED, val_fraction=VAL_FRACTION,
    resolution_eval_batch=RESOLUTION_EVAL_BATCH,
)

## 1. Datos

Los mismos conjuntos que `07_pdno_calor_2d.ipynb` (misma referencia, malla, $k_{\text{cap}}$, semilla y particion por condicion inicial), mas la extrapolacion temporal. Los campos exactos se generan en float64 y se pasan a `DTYPE` antes de que los vea cualquier modelo; los de float64 solo se usan para remuestrear (seccion 6).

In [ ]:
display(calor.load_data())

**Condiciones iniciales y evolucion.** Dos $u_0$ de entrenamiento y la solucion exacta en algunos tiempos de `TRAIN_TIMES`, cada $u_0$ y cada tiempo en su imagen.

In [ ]:
calor.initial_conditions()

## 2. PDNO guardado

El PDNO no se reentrena: se carga el `modelo.pt` que eligio `07_pdno_calor_2d.ipynb` y se comprueba que es de los mismos datos, en `DTYPE` y con la misma `TRAIN_CONFIG` (salvo el weight decay, que es un eje de su barrido), y que su error de prueba coincide con el de su `prueba.csv`. Si falta `modelo.pt`, la celda falla y dice que notebook correr. Se muestran su banda $(c,C)$, $\kappa$ y las $C_\alpha$ con que se entreno.

In [ ]:
resumen_pdno, class_C = calor.load_pdno()
display(resumen_pdno.to_frame())
display(class_C)

## 3. Barridos y seleccion

### 3.1 PDNO

El barrido de la MLP del simbolo de `07_pdno_calor_2d.ipynb` (su `barrido.csv`), de mejor a peor $L^2$ relativa de validacion; la corrida elegida es la primera.

In [ ]:
display(calor.pdno_sweep())

### 3.2 FNO

`FNOBaseline` envuelve el `FNO` de `neuraloperator` 2.0.0 con la interfaz del PDNO, `forward(u0, t)`, como en `10_benchmark_calor_1d.ipynb`: entrada $u_0(x)$ y $t$ constante, embedding de posicion `"grid"`, salida $u(t)$, sin `domain_padding`. En 2D `n_modes` es por eje y conserva $|k_i|\le$ `n_modes`$/2$: con $64$ el FNO ve la misma ventana $|k_i|\le32$ que las filas espectrales del PDNO. Cada corrida es `train_baseline` con `TRAIN_CONFIG` y weight decay $0$ (misma perdida, optimizador y early stopping que el PDNO, sin los terminos del simbolo); se elige la de **menor $L^2$ relativa de validacion**.

In [ ]:
calor.sweep_fno()

In [ ]:
barrido_fno, por_eje = calor.sweep_fno_results()
display(barrido_fno)
for tabla in por_eje:
    display(tabla)

In [ ]:
display(calor.select_fno().to_frame())

### 3.3 Curvas de entrenamiento

Las dos corridas elegidas, por epoca, en entrenamiento y validacion y en escala log, una imagen por termino. PDNO: perdida total y cada termino, leidos de `historia.csv` de su notebook (`perdida_pdno_*.png`). FNO: su unico termino, la $L^2$ relativa (`perdida_fno_rel_l2.png`). La linea punteada marca la epoca restaurada por el early stopping.

In [ ]:
calor.curves()

## 4. Prueba

Error relativo $L^2$ de cada condicion inicial de prueba y tiempo de `TEST_TIMES`, media y maximo por tiempo, para los dos modelos en la misma tabla y la misma figura. Los tiempos que no estan en `TRAIN_TIMES` van con circulos y lineas grises.

In [ ]:
display(calor.test())

**Campo.** Primera condicion inicial de prueba en algunos tiempos de `TEST_TIMES`: la exacta, la del PDNO y la del FNO (`campo_{exacta,pdno,fno}_t*.png`, con la misma escala de color) y el error de cada modelo (`campo_error_{pdno,fno}_t*.png`, con otra escala comun), cada campo y tiempo en su imagen.

In [ ]:
calor.field_figure()

## 5. Extrapolacion temporal

Las $u_0$ de prueba en `EXTRAPOLATION_TIMES` $\subset(0.95,3]$, fuera del rango de entrenamiento. Solo se reporta: no interviene en el entrenamiento ni en la seleccion.

In [ ]:
display(calor.extrapolation())

## 6. Invariancia de resolucion

Los dos modelos, entrenados en $128\times128$, sobre las mismas $u_0$ de prueba en `EVAL_RESOLUTIONS`, como en el 10 (`resample_fields` desde los campos float64 y luego a `DTYPE`): submuestreo hacia las mallas gruesas y relleno con ceros del espectro hacia las finas, eje por eje, que es la solucion exacta de la misma $u_0$. La celda comprueba que submuestrear los campos rellenados devuelve los originales. El PDNO evalua su simbolo en los modos de cada malla; el FNO usa los mismos `n_modes` y recalcula el embedding de posicion. En $256\times256$ se evalua con lotes de `RESOLUTION_EVAL_BATCH` pares: con $64$ no cabe en $6$ GB.

In [ ]:
display(calor.resolution_invariance())

## 7. Simbolo y certificacion del PDNO

Las secciones 7 y 8 de `07_pdno_calor_2d.ipynb` con el modelo cargado: simbolo efectivo frente a $|k|^2$ (`simbolo.png`) y $\bar a(k)/\langle k\rangle^m$ en la banda (`simbolo_banda.png`), su error relativo contra $|k|$ y sobre $(k_1,k_2)$, la certificacion de parabolicidad (Def. 2.3) y la clase $S^{m,\rho}$ (Def. 2.1). El FNO no tiene simbolo.

In [ ]:
display(calor.symbol())

In [ ]:
display(calor.certification())

In [ ]:
display(calor.symbol_class())

## 8. Modelo guardado

`save_fno_model` guarda el FNO elegido en `modelo_fno.pt` (su `FNOConfig`, la version de `neuraloperator`, la `OperatorTrainingConfig` y la misma entrada de datos que el PDNO). El PDNO ya esta guardado en su notebook. La ultima celda recarga el FNO y comprueba que da el mismo error de prueba.

In [ ]:
calor.save_fno()

In [ ]:
calor.check_saved_fno()